# Unit 06｜批量、多样性与约束

## Goal

在二维候选池中先过滤不可行候选，再比较纯 top-score 与兼顾多样性的批次。

本 Notebook 是确定性的人工教学实验，不是学习者已完成的研究，
也不是粘合剂实验结果。


## Setup

二维特征已处于 0–1 范围。可行性只使用候选输入，不读取隐藏目标。


In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import pairwise_distances

axis = np.linspace(0.0, 1.0, 9)
grid_a, grid_b = np.meshgrid(axis, axis)
X_all = np.column_stack([grid_a.ravel(), grid_b.ravel()])
candidate_ids = np.array([f"B{i:02d}" for i in range(len(X_all))])
oracle_values = (
    2.0 * np.sin(2.5 * X_all[:, 0])
    + 1.5 * np.cos(2.0 * X_all[:, 1])
    + X_all[:, 0] * X_all[:, 1]
)

def offline_oracle(global_index):
    return float(oracle_values[int(global_index)])

feasible_mask = (
    (X_all[:, 0] >= 0.125)
    & (X_all[:, 0] + X_all[:, 1] <= 1.45)
)
feasible_indices = np.flatnonzero(feasible_mask)
initial_indices = feasible_indices[
    np.linspace(0, len(feasible_indices) - 1, 7, dtype=int)
]
initial_values = np.array([
    offline_oracle(index) for index in initial_indices
])
pool_indices = np.setdiff1d(feasible_indices, initial_indices)
print("全部/可行/初始/候选:", len(X_all), len(feasible_indices), len(initial_indices), len(pool_indices))


全部/可行/初始/候选: 81 57 7 50


## Steps

按顺序执行。每个变量第一次出现时，先确认它的类型、形状和标签权限。


### 1. 训练 RF 集成并计算分歧

集成只读取已标注标签，候选标签保持隐藏。


In [2]:
predictions = []
for seed in [11, 22, 33, 44, 55]:
    member = RandomForestRegressor(
        n_estimators=100,
        max_features="sqrt",
        random_state=seed,
        n_jobs=1,
    )
    member.fit(X_all[initial_indices], initial_values)
    predictions.append(member.predict(X_all[pool_indices]))
matrix = np.vstack(predictions)
mean = matrix.mean(axis=0)
disagreement = matrix.std(axis=0)


### 2. 生成纯 top-score 批次

分数相同时按候选 ID 固定顺序。


In [3]:
batch_size = 6
top_order = np.lexsort((
    candidate_ids[pool_indices].astype(str),
    -disagreement,
))
top_local = top_order[:batch_size]
top_global = pool_indices[top_local]

random_rng = np.random.default_rng(2026)
random_local = random_rng.choice(
    len(pool_indices),
    size=batch_size,
    replace=False,
)
random_global = pool_indices[random_local]


### 3. 生成 score + diversity 批次

每次选择与已选批次距离较远、同时分数较高的候选。


In [4]:
score_range = np.ptp(disagreement)
normalized_score = (
    (disagreement - disagreement.min())
    / max(score_range, 1e-12)
)
def select_with_tie_break(score, ids):
    order = np.lexsort((ids.astype(str), -np.asarray(score)))
    return int(order[0])

selected_local = [
    select_with_tie_break(
        normalized_score,
        candidate_ids[pool_indices],
    )
]
diversity_weight = 0.8

while len(selected_local) < batch_size:
    distances = pairwise_distances(
        X_all[pool_indices],
        X_all[pool_indices[selected_local]],
    ).min(axis=1)
    distance_range = np.ptp(distances)
    normalized_distance = (
        (distances - distances.min())
        / max(distance_range, 1e-12)
    )
    combined = (
        normalized_score
        + diversity_weight * normalized_distance
    )
    combined[selected_local] = -np.inf
    selected_local.append(
        select_with_tie_break(
            combined,
            candidate_ids[pool_indices],
        )
    )

diverse_local = np.array(selected_local)
diverse_global = pool_indices[diverse_local]


### 4. 比较批内距离并建立审核表

只比较候选特征距离；proposed 表不包含 Oracle 标签。


In [5]:
def batch_distance_summary(global_indices):
    distances = pairwise_distances(X_all[global_indices])
    upper = distances[
        np.triu_indices(len(global_indices), k=1)
    ]
    return {
        "minimum": float(upper.min()),
        "mean": float(upper.mean()),
    }

distance_table = pd.DataFrame([
    {
        "strategy": "random",
        **batch_distance_summary(random_global),
    },
    {
        "strategy": "top_score",
        **batch_distance_summary(top_global),
    },
    {
        "strategy": "score_plus_diversity",
        **batch_distance_summary(diverse_global),
    },
])

proposed = pd.DataFrame({
    "global_index": diverse_global,
    "candidate_id": candidate_ids[diverse_global],
    "feature_a": X_all[diverse_global, 0],
    "feature_b": X_all[diverse_global, 1],
    "prediction_mean": mean[diverse_local],
    "disagreement": disagreement[diverse_local],
    "feasible": feasible_mask[diverse_global],
    "review_status": "proposed",
    "rejection_reason": "",
})
print(distance_table.round(3).to_string(index=False))
print(proposed.round(3).to_string(index=False))


            strategy  minimum  mean
              random    0.125 0.350
           top_score    0.125 0.294
score_plus_diversity    0.125 0.400
 global_index candidate_id  feature_a  feature_b  prediction_mean  disagreement  feasible review_status rejection_reason
           58          B58      0.500      0.750            1.896         0.061      True      proposed                 
           30          B30      0.375      0.375            2.883         0.060      True      proposed                 
           46          B46      0.125      0.625            2.203         0.055      True      proposed                 
           31          B31      0.500      0.375            2.969         0.060      True      proposed                 
            4          B04      0.500      0.000            2.677         0.042      True      proposed                 
           49          B49      0.500      0.625            2.622         0.058      True      proposed                 


### 5. 审核后才调用 Oracle，并保留失败/拒绝

这里用确定性规则模拟人工审核与一次测量失败；真实项目必须由授权人员操作。


In [6]:
reviewed = proposed.copy()
reviewed["review_status"] = "approved"
reviewed.loc[
    reviewed.index[-1],
    ["review_status", "rejection_reason"],
] = ["rejected", "教学模拟：设备档期不允许"]

failed_index = reviewed.index[0]
reviewed["observed_y"] = np.nan
reviewed["failure_reason"] = ""
for row_index, row in reviewed.iterrows():
    if row["review_status"] != "approved":
        continue
    if row_index == failed_index:
        reviewed.loc[
            row_index,
            ["review_status", "failure_reason"],
        ] = ["failed", "教学模拟：测量失败"]
        continue

    # 标签揭示线：只有 approved 且实际完成的候选才查询。
    reviewed.loc[row_index, "observed_y"] = offline_oracle(
        int(row["global_index"])
    )
    reviewed.loc[row_index, "review_status"] = "completed"

print(reviewed.round(3).to_string(index=False))


 global_index candidate_id  feature_a  feature_b  prediction_mean  disagreement  feasible review_status rejection_reason  observed_y failure_reason
           58          B58      0.500      0.750            1.896         0.061      True        failed                          NaN      教学模拟：测量失败
           30          B30      0.375      0.375            2.883         0.060      True     completed                        2.850               
           46          B46      0.125      0.625            2.203         0.055      True     completed                        1.166               
           31          B31      0.500      0.375            2.969         0.060      True     completed                        3.183               
            4          B04      0.500      0.000            2.677         0.042      True     completed                        3.398               
           49          B49      0.500      0.625            2.622         0.058      True      rejected     教学模拟

## Checks

这些断言检查形状、预算和无重复等机械条件；通过断言不代表研究结论已经成立。


In [7]:
assert len(np.unique(top_global)) == batch_size
assert len(np.unique(diverse_global)) == batch_size
assert len(np.unique(random_global)) == batch_size
assert feasible_mask[top_global].all()
assert feasible_mask[diverse_global].all()
assert feasible_mask[random_global].all()
assert proposed["review_status"].eq("proposed").all()
assert "observed_y" not in proposed.columns
means = distance_table.set_index("strategy")["mean"]
assert means["score_plus_diversity"] >= means["top_score"]
assert set(reviewed["review_status"]) == {
    "completed", "failed", "rejected"
}
assert reviewed.loc[
    reviewed["review_status"] == "completed", "observed_y"
].notna().all()
assert reviewed.loc[
    reviewed["review_status"] != "completed", "observed_y"
].isna().all()
print("Unit 06 checks passed.")


Unit 06 checks passed.


## Next Steps

改变多样性权重并记录批次变化。Unit 07 将在相同采集接口下替换 RF/MLP 等代理模型。
